In [1]:
from langchain_core.documents import Document

In [ ]:
doc=Document(page_content="This is the content for RAG",
 metadata={
    "source": "example.com",
    "author": "John Doe",
    "date": "2021-01-01"
    })

print(doc.page_content)
print(doc.metadata)

In [ ]:
### Text Loader

from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/textfiles/ishan.txt")
docs = loader.load()
print(docs)




In [ ]:
### PDF Loader
from langchain_community.document_loaders import PyMuPDFLoader
loader=PyMuPDFLoader("../data/pdf/fire_alarm.pdf")  
docs=loader.load()
print(docs)

In [ ]:
##Directory Loader for TExt Files
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader("../data/textfiles",
glob="*.txt",
loader_cls=TextLoader,
loader_kwargs={"encoding":"utf-8"}
)
docs=dir_loader.load()
print(docs)

In [ ]:
##Directory Loader for TExt Files
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader("../data/pdf",
glob="*.pdf",
loader_cls=PyMuPDFLoader,
)
docs=dir_loader.load()
print(docs)

[Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '../data/pdf/fire_alarm.pdf', 'file_path': '../data/pdf/fire_alarm.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': 'fire_alarm_system_IshanBaidya', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='FIRE ALARM SYSTEM\n \n \nComponents used: \n1.\u200b Arduino UNO \n2.\u200b Smoke Sensor / MQ-2 Gas module \n3.\u200b Red and Green LED \n4.\u200b Breadboard and Wires \n5.\u200b Buzzer \n6.\u200b Resistor \n \nNormal Condition \n1.\u200b MQ-2 sensor detects clean air. \n2.\u200b Analog value remains below threshold. \n3.\u200b Arduino keeps: Green LED ON, Red LED OFF, Buzzer OFF \nFire/Smoke Detected \n1.\u200b Smoke concentration increases. \n2.\u200b MQ-2 sensor output voltage rises. \n3.\u200b Arduino reads higher analog values \n4.\u200b When value exceeds predefined threshold:Red LE

In [ ]:
### PDF Chunking helpers
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def load_pdf(path: str) -> list[Document]:
    """Load a single PDF into one Document per page."""
    return PyMuPDFLoader(path).load()


def load_pdf_dir(dir_path: str, glob: str = "*.pdf") -> list[Document]:
    """Load every PDF in a directory."""
    return DirectoryLoader(dir_path, glob=glob, loader_cls=PyMuPDFLoader).load()


def chunk_documents(
    docs: list[Document],
    chunk_size: int = 800,
    chunk_overlap: int = 100,
) -> list[Document]:
    """Split loaded documents into overlapping chunks for RAG.

    Metadata from the source page is preserved on every chunk, and a
    `chunk_id` is added so chunks stay traceable back to their document.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    chunks = splitter.split_documents(docs)
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
    return chunks


def chunk_pdf(
    path: str,
    chunk_size: int = 800,
    chunk_overlap: int = 100,
) -> list[Document]:
    """Load a single PDF and return its chunks."""
    return chunk_documents(load_pdf(path), chunk_size, chunk_overlap)


def chunk_pdf_dir(
    dir_path: str,
    chunk_size: int = 800,
    chunk_overlap: int = 100,
) -> list[Document]:
    """Load all PDFs in a directory and return their chunks."""
    return chunk_documents(load_pdf_dir(dir_path), chunk_size, chunk_overlap)

In [ ]:
### Try the chunker on the fire alarm PDF
chunks = chunk_pdf("../data/pdf/fire_alarm.pdf", chunk_size=500, chunk_overlap=80)

print(f"Total chunks: {len(chunks)}")
for c in chunks[:3]:
    print("-" * 60)
    print("page:", c.metadata.get("page"), "| chunk_id:", c.metadata.get("chunk_id"))
    print("chars:", len(c.page_content))
    print(c.page_content[:200], "...")